# KenLM N-gram Integration with Conformer for Newari ASR

This notebook integrates a KenLM n-gram language model with a Conformer-CTC model for improved Newari speech recognition.

## Setup Overview:
- **Model**: Conformer-CTC-BPE
- **Language Model**: KenLM 5-gram
- **Corpus Size**: 40k sentences
- **Integration Methods**: 
  1. **pyctcdecode** (Primary): Uses external beam search decoder with LM
  2. **NeMo Native** (Fallback): Uses NeMo's built-in n-gram support

## Installation Notes:
This notebook includes multiple installation methods for KenLM:
1. **Full build from source** (recommended but requires build tools)
2. **Alternative pip installation** (if build fails)
3. **NeMo native method** (works without KenLM Python bindings)

The notebook will automatically try different methods and fall back to alternatives if needed.

## 1. Install Required Packages

In [1]:
# Install system dependencies first
!apt-get update -qq
!apt-get install -y build-essential cmake libboost-all-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev

In [2]:
# Clone and build KenLM properly
!git clone https://github.com/kpu/kenlm.git /tmp/kenlm
!cd /tmp/kenlm && mkdir -p build && cd build && cmake .. && make -j4

# Install KenLM Python bindings
!pip install /tmp/kenlm

In [3]:
# Install pyctcdecode for CTC decoding with language model
!pip install pyctcdecode

# Install other necessary packages
!pip install nemo_toolkit['asr']==2.1.0
!pip install pynini  # For text normalization if needed

In [ ]:
!pip install soundfile librosa jiwer

In [5]:
# Fix NumPy 2.0 compatibility issues
import numpy as np
if not hasattr(np, 'sctypes'):
    # Recreate np.sctypes for backward compatibility
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str]
    }

In [6]:
# Verify KenLM installation
import kenlm
print("✓ KenLM installed successfully")
print(f"KenLM version: {kenlm.__file__}")

### Alternative Installation (If above fails)

In [8]:
# If the above installation fails, try this alternative:
# This installs pre-built binaries

# !pip install --upgrade pip setuptools wheel
# !pip install kenlm --verbose

# Or build from source with specific flags:
# !git clone https://github.com/kpu/kenlm.git /tmp/kenlm
# !cd /tmp/kenlm && python setup.py install --force

print("Alternative installation methods available (commented out).")

### Troubleshooting KenLM Installation

In [9]:
# Test if KenLM is working
try:
    import kenlm
    print("✓ KenLM module imported successfully")
    
    # Check if Model class exists
    if hasattr(kenlm, 'Model'):
        print("✓ kenlm.Model class is available")
    else:
        print("✗ kenlm.Model class NOT found")
        print("Available attributes:", dir(kenlm))
        
except ImportError as e:
    print(f"✗ Failed to import kenlm: {e}")
    print("\nPlease run the alternative installation above.")

## 2. Import Libraries

In [10]:
import os
import nemo.collections.asr as nemo_asr
from pyctcdecode import build_ctcdecoder
import kenlm
import numpy as np
from pathlib import Path
import json
import torch

print("Libraries imported successfully!")

## 3. Define Paths and Parameters

In [12]:
# Model and corpus paths
# EDIT: Update to your own file paths
MODEL_PATH = "/kaggle/input/datasets/nwacha-muna/conformer-ctc-bpe-finetune/Conformer-CTC-BPE-Finetune (5).nemo"
# EDIT: Update to your own file paths
CORPUS_PATH = "/kaggle/input/datasets/nwacha-muna/newaritokenizer/cleaned_newari.txt"

# Output paths
WORKING_DIR = "/kaggle/working"
ARPA_PATH = os.path.join(WORKING_DIR, "newari_5gram.arpa")
BINARY_LM_PATH = os.path.join(WORKING_DIR, "newari_5gram.binary")

# LM parameters
NGRAM_ORDER = 5  # 5-gram model
PRUNE_THRESHOLD = "0 0 1"  # Pruning: keep all 1,2-grams, prune 3+grams with count=1

# Beam search parameters
BEAM_WIDTH = 128
ALPHA = 0.4  # Language model weight
BETA = 1.0  # Word insertion bonus

print(f"Working directory: {WORKING_DIR}")
print(f"N-gram order: {NGRAM_ORDER}")
print(f"Beam width: {BEAM_WIDTH}")

## 4. Verify Input Files

In [13]:
# Check if model exists
if os.path.exists(MODEL_PATH):
    print(f"✓ Model found: {MODEL_PATH}")
else:
    print(f"✗ Model not found: {MODEL_PATH}")

# Check if corpus exists and show stats
if os.path.exists(CORPUS_PATH):
    print(f"✓ Corpus found: {CORPUS_PATH}")
    with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"  Total lines: {len(lines)}")
    print(f"  Sample text: {lines[0].strip()[:100]}...")
else:
    print(f"✗ Corpus not found: {CORPUS_PATH}")

## 5. Preprocess Text Corpus

In [14]:
def preprocess_corpus(input_path, output_path):
    """
    Preprocess text corpus for KenLM training.
    - Remove empty lines
    - Convert to lowercase (optional, depends on your needs)
    - Remove extra whitespace
    """
    processed_lines = []
    
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:  # Skip empty lines
                # Clean whitespace
                line = ' '.join(line.split())
                processed_lines.append(line)
    
    # Write processed corpus
    with open(output_path, 'w', encoding='utf-8') as f:
        for line in processed_lines:
            f.write(line + '\n')
    
    return len(processed_lines)

# Preprocess corpus
PROCESSED_CORPUS = os.path.join(WORKING_DIR, "processed_newari.txt")
num_sentences = preprocess_corpus(CORPUS_PATH, PROCESSED_CORPUS)

print(f"Processed {num_sentences} sentences")
print(f"Saved to: {PROCESSED_CORPUS}")

In [15]:
# Load the Conformer model
print("Loading Conformer model...")
asr_model = nemo_asr.models.EncDecCTCModelBPE.restore_from(MODEL_PATH)

In [16]:
# Process text corpus into BPE tokens for LM training
print("Converting text corpus to BPE tokens...")

# Load processed corpus
with open("/kaggle/working/processed_newari.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Processing {len(lines)} lines...")

# Convert to BPE tokens
bpe_lines = []
skipped = 0

for i, line in enumerate(lines):
    line = line.strip()
    if not line:  # Skip empty lines
        skipped += 1
        continue
    
    try:
        # Convert text to BPE tokens using the model's tokenizer
        tokens = asr_model.tokenizer.text_to_tokens(line)
        
        # Remove special tokens
        tokens = [t for t in tokens if t not in ['<s>', '</s>', '<unk>', '<pad>']]
        
        if tokens:  # Only add if tokens exist
            bpe_lines.append(" ".join(tokens))
    except Exception as e:
        print(f"Error on line {i}: {e}")
        skipped += 1
        continue
    
    # Progress indicator
    if (i + 1) % 5000 == 0:
        print(f"  Processed {i + 1}/{len(lines)} lines...")

# Write BPE tokenized corpus
BPE_CORPUS_PATH = "/kaggle/working/newari_bpe_lm.txt"
with open(BPE_CORPUS_PATH, "w", encoding="utf-8") as out:
    for line in bpe_lines:
        out.write(line + "\n")

print(f"\n✓ BPE tokenization complete!")
print(f"  Original lines: {len(lines)}")
print(f"  Processed lines: {len(bpe_lines)}")
print(f"  Skipped: {skipped}")
print(f"  Saved to: {BPE_CORPUS_PATH}")

# Show sample
print(f"\nSample BPE tokens:")
for i in range(min(3, len(bpe_lines))):
    print(f"  {bpe_lines[i][:100]}...")

## 6. Train KenLM Language Model

In [18]:
# Build ARPA format language model
print("Training KenLM language model...")
print(f"This may take a few minutes for {num_sentences} sentences...\n")

# KenLM command
# --discount_fallback: handle unseen n-grams gracefully
# -S 80%: use 80% of available RAM
# KenLM command with full path
kenlm_cmd = f"""
/tmp/kenlm/build/bin/lmplz -o {NGRAM_ORDER} \
      --discount_fallback \
      -S 80% \
      --prune {PRUNE_THRESHOLD} \
      < {PROCESSED_CORPUS} \
      > {ARPA_PATH}
"""

# !{kenlm_cmd}

!{kenlm_cmd}

print(f"\n✓ ARPA model saved to: {ARPA_PATH}")

## 7. Convert ARPA to Binary Format (for faster loading)

In [21]:
!/kaggle/working/kenlm/build/bin/build_binary /kaggle/working/newari_5gram.arpa /kaggle/working/newari_5gram.binary


## 8. Load Conformer Model and Extract Vocabulary

In [23]:


# Set to eval mode
asr_model.eval()

print("✓ Model loaded successfully")
print(f"  Model type: {type(asr_model).__name__}")
print(f"  Vocabulary size: {len(asr_model.decoder.vocabulary)}")

In [24]:
# Extract vocabulary from the model
# Convert ListConfig to regular Python list
vocabulary = list(asr_model.decoder.vocabulary)

# Get the tokenizer
tokenizer = asr_model.tokenizer

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Sample tokens: {vocabulary[:20]}")

# Save vocabulary for reference
VOCAB_PATH = os.path.join(WORKING_DIR, "vocabulary.json")
with open(VOCAB_PATH, 'w', encoding='utf-8') as f:
    json.dump(vocabulary, f, ensure_ascii=False, indent=2)

print(f"\n✓ Vocabulary saved to: {VOCAB_PATH}")

## 9. Build CTC Decoder with Language Model

In [25]:
# Try to build the decoder with LM integration
print("Building CTC decoder with language model...")

try:
    # IMPORTANT: Use vocabulary EXACTLY as extracted from model
    # The model outputs logits for all tokens in its vocabulary, including special tokens
    # pyctcdecode will handle filtering these during decoding
    decoder = build_ctcdecoder(
        labels=vocabulary,  # Use full vocabulary without filtering
        kenlm_model_path=BINARY_LM_PATH,
        alpha=ALPHA,
        beta=BETA
    )
    
    print("✓ Decoder built successfully with pyctcdecode")
    print(f"  Vocabulary size: {len(vocabulary)}")
    print(f"  Alpha (LM weight): {ALPHA}")
    print(f"  Beta (word bonus): {BETA}")
    DECODER_METHOD = "pyctcdecode"
    
except Exception as e:
    print(f"⚠ Failed to build pyctcdecode decoder: {e}")
    print("\nFalling back to NeMo native beam search...")
    
    # Method 2: Use NeMo's native beam search (fallback)
    decoder = None
    DECODER_METHOD = "nemo_native"
    print("✓ Will use NeMo's native beam search decoder")

## 10. Test the Integration

In [27]:
def get_logits_from_audio(audio_path):
    """
    Extract logits from audio file using NeMo model.
    
    Args:
        audio_path: path to audio file
    
    Returns:
        numpy array of logits (time_steps, vocab_size)
    """
    import soundfile as sf
    
    # Load audio with soundfile
    audio, sr = sf.read(audio_path)
    
    # Resample if needed using scipy instead of librosa
    if sr != 16000:
        try:
            from scipy import signal
            # Calculate resampling ratio
            num_samples = int(len(audio) * 16000 / sr)
            audio = signal.resample(audio, num_samples)
        except ImportError:
            # Fallback to librosa if scipy not available
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    
    # Process through model to get logits
    with torch.no_grad():
        # Convert to tensor and move to model's device
        audio_tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
        if torch.cuda.is_available():
            audio_tensor = audio_tensor.cuda()
        audio_len = torch.tensor([audio_tensor.shape[1]], dtype=torch.int32)
        
        # Get encoder output
        if hasattr(asr_model, 'preprocessor'):
            processed_signal, processed_len = asr_model.preprocessor(
                input_signal=audio_tensor,
                length=audio_len
            )
        else:
            processed_signal = audio_tensor
            processed_len = audio_len
        
        # Get encoder output
        if hasattr(asr_model, 'encoder'):
            encoded, encoded_len = asr_model.encoder(
                audio_signal=processed_signal,
                length=processed_len
            )
        else:
            encoded = processed_signal
        
        # Get decoder logits
        logits = asr_model.decoder(encoder_output=encoded)
        logits = logits.cpu().numpy()[0]  # Remove batch dimension
    
    return logits

def decode_with_lm(logits, decoder, beam_width=100):
    """Decode with LM and remove special tokens."""
    if decoder is not None:
        text = decoder.decode(logits, beam_width=beam_width)
    else:
        text = decode_greedy(logits)
    
    # Remove special tokens
    text = text.replace('<s>', '').replace('</s>', '').replace('<unk>', '')
    text = text.strip()
    return text

# def decode_without_lm(logits, vocabulary):
    
#    """Greedy decode and remove special tokens."""
#     # indices = np.argmax(logits, axis=-1)
    
#     collapsed = []
#     prev_idx = -1
#     for idx in indices:
#         if idx != 0 and idx != prev_idx:
#             collapsed.append(idx)
#         prev_idx = idx
    
#     if len(collapsed) == 0:
#         return ""
    
#     text = tokenizer.ids_to_text(collapsed)
    
#     # Remove special tokens
#     text = text.replace('<s>', '').replace('</s>', '').replace('<unk>', '')
#     text = text.strip()
    
#     return text


def decode_without_lm(logits, vocabulary):
    """Greedy decode and remove special tokens."""
    indices = np.argmax(logits, axis=-1)
    
    collapsed = []
    prev_idx = -1
    for idx in indices:
        if idx != 0 and idx != prev_idx:
            collapsed.append(idx)
        prev_idx = idx
    
    if len(collapsed) == 0:
        return ""
    
    text = ''.join([vocabulary[i] if i < len(vocabulary) else '' for i in collapsed])
    
    # Remove special tokens
    text = text.replace('<s>', '').replace('</s>', '').replace('<unk>', '')
    text = text.strip()
    
    return text


print("Decoding functions defined.")
print(f"Active decoder method: {DECODER_METHOD}")


In [29]:
import librosa
import soundfile as sf
import tempfile

def preprocess_audio_to_mono(audio_path, output_dir="/tmp"):
    """
    Convert stereo audio to mono.
    
    Args:
        audio_path: input audio file path
        output_dir: directory to save mono audio
    
    Returns:
        path to mono audio file
    """
    # Load audio (librosa automatically converts to mono)
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    
    # Save as mono
    output_path = os.path.join(output_dir, f"mono_{os.path.basename(audio_path)}")
    sf.write(output_path, audio, sr)
    return output_path

print("Audio preprocessing function ready.")


## 11. Batch Transcription (Optional)

In [32]:
def batch_transcribe(audio_files, use_lm=True, beam_width=BEAM_WIDTH):
    """
    Transcribe multiple audio files.
    
    Args:
        audio_files: list of audio file paths
        use_lm: whether to use language model
        beam_width: beam width for LM decoding
    
    Returns:
        list of transcriptions
    """
    transcriptions = []
    
    for i, audio_path in enumerate(audio_files):
        print(f"Processing {i+1}/{len(audio_files)}: {audio_path}")
        try:
            text = transcribe_audio(audio_path, use_lm, beam_width)
            transcriptions.append(text)
        except Exception as e:
            print(f"  Error: {e}")
            transcriptions.append("")
    
    return transcriptions

print("Batch transcription function ready.")

## 12. Hyperparameter Tuning (Optional)

In [33]:
def tune_lm_parameters(audio_path, reference_text, 
                       alpha_range=[0.3, 0.5, 0.7, 1.0],
                       beta_range=[0.5, 1.0, 1.5, 2.0]):
    """
    Grid search for optimal alpha and beta parameters.
    
    Args:
        audio_path: path to test audio
        reference_text: ground truth transcription
        alpha_range: list of alpha values to try
        beta_range: list of beta values to try
    """
    from jiwer import wer
    
    best_wer = float('inf')
    best_params = None
    results = []
    
    # Get logits once
    logits = get_logits_from_audio(audio_path)
    
    for alpha in alpha_range:
        for beta in beta_range:
            # Build decoder with new params
            test_decoder = build_ctcdecoder(
                labels=vocabulary,
                kenlm_model_path=BINARY_LM_PATH,
                alpha=alpha,
                beta=beta
            )
            
            # Decode
            hypothesis = decode_with_lm(logits, test_decoder, BEAM_WIDTH)
            
            # Calculate WER
            error = wer(reference_text, hypothesis)
            
            results.append({
                'alpha': alpha,
                'beta': beta,
                'wer': error,
                'hypothesis': hypothesis
            })
            
            if error < best_wer:
                best_wer = error
                best_params = (alpha, beta)
            
            print(f"Alpha={alpha}, Beta={beta}: WER={error:.4f}")
    
    print(f"\nBest parameters: Alpha={best_params[0]}, Beta={best_params[1]}")
    print(f"Best WER: {best_wer:.4f}")
    
    return results, best_params

print("Hyperparameter tuning function ready.")
print("Note: Install jiwer first: !pip install jiwer")

## 13. Validation: Calculate WER/CER

In [35]:
from jiwer import wer, cer
import pandas as pd
from tqdm import tqdm

def evaluate_model(test_manifest_path, use_lm=True, beam_width=BEAM_WIDTH, max_samples=None):
    """
    Evaluate model on test set and calculate WER/CER.
    
    Args:
        test_manifest_path: path to test manifest JSON file
                           Each line should be: {"audio_filepath": "...", "text": "..."}
        use_lm: whether to use language model
        beam_width: beam width for decoding
        max_samples: maximum number of samples to evaluate (None for all)
    
    Returns:
        Dictionary with WER, CER, and detailed results
    """
    import json
    import soundfile as sf
    
    # Load test manifest
    test_data = []
    with open(test_manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            test_data.append(json.loads(line.strip()))
    
    if max_samples:
        test_data = test_data[:max_samples]
    
    print(f"Evaluating on {len(test_data)} samples...")
    print(f"Using LM: {use_lm}")
    print(f"Beam width: {beam_width if use_lm else 'N/A (greedy)'}")
    
    references = []
    hypotheses = []
    results = []
    
    # Create temp directory for mono audio
    temp_dir = tempfile.mkdtemp()
    
    # Process each sample
    for item in tqdm(test_data, desc="Processing"):
        audio_path = item['audio_filepath']
        reference = item['text']
        
        try:
            # Convert to mono
            mono_audio_path = preprocess_audio_to_mono(audio_path, temp_dir)
            
            if use_lm:
                # For LM decoding, we need logits - use forward pass
                # Read audio
                audio, sr = sf.read(mono_audio_path)
                audio_tensor = torch.tensor(audio).unsqueeze(0).to(asr_model.device)
                audio_len = torch.tensor([len(audio)]).to(asr_model.device)
                
                # Get features
                features, features_len = asr_model.preprocessor(input_signal=audio_tensor, length=audio_len)
                
                # Get encoder output
                encoded, encoded_len = asr_model.encoder(audio_signal=features, length=features_len)
                
                # Get logits
                logits = asr_model.decoder(encoder_output=encoded)
                logits = torch.log_softmax(logits, dim=-1)
                logits = logits[0].detach().cpu().numpy()
                
                # Decode with LM
                hypothesis = decode_with_lm(logits, decoder, beam_width)
            else:
                # For greedy decoding, just use model's transcribe
                hypothesis = asr_model.transcribe([mono_audio_path])[0]
            
            references.append(reference)
            hypotheses.append(hypothesis)
            
            # Calculate per-sample metrics
            sample_wer = wer(reference, hypothesis)
            sample_cer = cer(reference, hypothesis)
            
            results.append({
                'audio': audio_path,
                'reference': reference,
                'hypothesis': hypothesis,
                'wer': sample_wer,
                'cer': sample_cer
            })
            
        except Exception as e:
            print(f"\nError processing {audio_path}: {e}")
            continue
    
    # Calculate overall metrics
    if len(references) > 0:
        overall_wer = wer(references, hypotheses)
        overall_cer = cer(references, hypotheses)
    else:
        overall_wer = 1.0
        overall_cer = 1.0
    
    # Cleanup temp directory
    import shutil
    shutil.rmtree(temp_dir, ignore_errors=True)
    
    return {
        'wer': overall_wer,
        'cer': overall_cer,
        'num_samples': len(results),
        'detailed_results': results
    }

print("Evaluation function ready (with mono audio conversion and fixed logits extraction).")

In [36]:
def fix_manifest_paths(input_manifest, output_manifest, correct_base_path):
    """
    Fix audio paths in manifest file.
    
    Args:
        input_manifest: original manifest path
        output_manifest: output manifest path
        correct_base_path: correct base directory for audio files
    """
    import json
    
    # Check if input file exists and has content
    if not os.path.exists(input_manifest):
        print(f"✗ Input manifest not found: {input_manifest}")
        return False
    
    file_size = os.path.getsize(input_manifest)
    print(f"Input manifest size: {file_size} bytes")
    
    if file_size == 0:
        print("✗ Input manifest is empty!")
        return False
    
    # Read and process
    line_count = 0
    try:
        with open(input_manifest, 'r', encoding='utf-8') as f_in:
            with open(output_manifest, 'w', encoding='utf-8') as f_out:
                for line in f_in:
                    if line.strip():
                        try:
                            data = json.loads(line.strip())
                            
                            # Get just the filename
                            if 'audio_filepath' in data:
                                filename = os.path.basename(data['audio_filepath'])
                                # Build correct path
                                data['audio_filepath'] = os.path.join(correct_base_path, filename)
                                
                                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
                                line_count += 1
                        except json.JSONDecodeError as e:
                            print(f"⚠ Skipped invalid JSON line: {e}")
                            continue
        
        output_size = os.path.getsize(output_manifest)
        print(f"✓ Fixed manifest created: {output_manifest}")
        print(f"  Lines processed: {line_count}")
        print(f"  Output size: {output_size} bytes")
        return True
        
    except Exception as e:
        print(f"✗ Error processing manifest: {e}")
        return False

# First, find the correct manifest path
print("Searching for manifest files...")
import glob

# Search for manifest files
# EDIT: Update to your own file paths
manifests = glob.glob("/kaggle/input/datasets/nwacha-muna/final-data/*.json")
print(f"Found manifest files:")
for m in manifests:
    size = os.path.getsize(m)
    print(f"  - {m} ({size} bytes)")

# Use the correct validation manifest if it exists
# EDIT: Update to your own file paths
if os.path.exists("/kaggle/input/datasets/nwacha-muna/final-data/nemo_test.json"):
# EDIT: Update to your own file paths
    TEST_MANIFEST = "/kaggle/input/datasets/nwacha-muna/final-data/nemo_test.json"
    print(f"\n✓ Using validation manifest: {TEST_MANIFEST}")
else:
    print("\n✗ Validation manifest not found in expected location")

In [37]:
# Now fix the manifest
# EDIT: Update to your own file paths
CORRECT_AUDIO_PATH = "/kaggle/input/datasets/nwacha-muna/final-data/audio/audio"
FIXED_MANIFEST = "/kaggle/working/fixed_val_manifest.json"

print(f"Input manifest: {TEST_MANIFEST}")
print(f"Correct audio path: {CORRECT_AUDIO_PATH}")
print(f"Output manifest: {FIXED_MANIFEST}\n")

success = fix_manifest_paths(TEST_MANIFEST, FIXED_MANIFEST, CORRECT_AUDIO_PATH)

if success:
    # Verify a sample file exists
    print("\nVerifying sample file...")
    import json
    with open(FIXED_MANIFEST, 'r', encoding='utf-8') as f:
        sample = json.loads(f.readline())
        sample_path = sample['audio_filepath']
        print(f"Sample path: {sample_path}")
        print(f"File exists: {os.path.exists(sample_path)}")
    
    # Update TEST_MANIFEST for evaluation
    TEST_MANIFEST = FIXED_MANIFEST
    print(f"\n✓ Ready to use fixed manifest for evaluation")
else:
    print("✗ Failed to create fixed manifest")

### Run Validation (Example Usage)

In [39]:
# Example: Set your test manifest path
# EDIT: Update to your own file paths
# TEST_MANIFEST = "/kaggle/input/datasets/nwacha-muna/final-data/nemo_test.json"

# Uncomment and run when you have a test set:

# Evaluate WITHOUT language model (baseline)
print("=" * 60)
print("BASELINE: Greedy CTC Decoding (No LM)")
print("=" * 60)
baseline_results = evaluate_model(
    TEST_MANIFEST,
    use_lm=False,
)

print(f"\nBaseline Results:")
print(f"  WER: {baseline_results['wer']:.4f} ({baseline_results['wer']*100:.2f}%)")
print(f"  CER: {baseline_results['cer']:.4f} ({baseline_results['cer']*100:.2f}%)")
print(f"  Samples: {baseline_results['num_samples']}")

# Evaluate WITH language model
print("\n" + "=" * 60)
print("WITH LANGUAGE MODEL: Beam Search + KenLM")
print("=" * 60)
lm_results = evaluate_model(
    TEST_MANIFEST,
    use_lm=True,
    beam_width=BEAM_WIDTH,
)

print(f"\nLanguage Model Results:")
print(f"  WER: {lm_results['wer']:.4f} ({lm_results['wer']*100:.2f}%)")
print(f"  CER: {lm_results['cer']:.4f} ({lm_results['cer']*100:.2f}%)")
print(f"  Samples: {lm_results['num_samples']}")

# Calculate improvement
wer_improvement = (baseline_results['wer'] - lm_results['wer']) / baseline_results['wer'] * 100
cer_improvement = (baseline_results['cer'] - lm_results['cer']) / baseline_results['cer'] * 100

print("\n" + "=" * 60)
print("IMPROVEMENT SUMMARY")
print("=" * 60)
print(f"WER Improvement: {wer_improvement:+.2f}%")
print(f"CER Improvement: {cer_improvement:+.2f}%")


print("\nValidation code ready. Set TEST_MANIFEST path and uncomment to run.")

### Compare Different Beam Widths

In [49]:
def compare_beam_widths(test_manifest_path, beam_widths=[10, 50, 100, 200], max_samples=50):
    """
    Compare WER/CER across different beam widths.
    
    Args:
        test_manifest_path: path to test manifest
        beam_widths: list of beam widths to test
        max_samples: number of samples to test (for speed)
    """
    results_summary = []
    
    for bw in beam_widths:
        print(f"\nTesting beam width: {bw}")
        print("-" * 40)
        
        result = evaluate_model(
            test_manifest_path,
            use_lm=True,
            beam_width=bw,
            max_samples=max_samples
        )
        
        results_summary.append({
            'beam_width': bw,
            'wer': result['wer'],
            'cer': result['cer'],
            'samples': result['num_samples']
        })
        
        print(f"WER: {result['wer']:.4f}")
        print(f"CER: {result['cer']:.4f}")
    
    # Display summary
    print("\n" + "=" * 60)
    print("BEAM WIDTH COMPARISON")
    print("=" * 60)
    
    df = pd.DataFrame(results_summary)
    print(df.to_string(index=False))
    
    # Find best
    best_idx = df['wer'].idxmin()
    print(f"\nBest beam width: {df.loc[best_idx, 'beam_width']}")
    print(f"  WER: {df.loc[best_idx, 'wer']:.4f}")
    print(f"  CER: {df.loc[best_idx, 'cer']:.4f}")
    
    return df

print("Beam width comparison function ready.")

### Analyze Errors (Sample-level Analysis)

In [50]:
def analyze_errors(results_dict, top_n=10, save_path=None):
    """
    Analyze and display worst performing samples.
    
    Args:
        results_dict: output from evaluate_model function
        top_n: number of worst samples to display
        save_path: optional path to save detailed error analysis
    """
    detailed = results_dict['detailed_results']
    
    # Convert to DataFrame for easy analysis
    df = pd.DataFrame(detailed)
    
    # Sort by WER (worst first)
    df_sorted = df.sort_values('wer', ascending=False)
    
    print("=" * 80)
    print(f"TOP {top_n} WORST PREDICTIONS (by WER)")
    print("=" * 80)
    
    for i, row in df_sorted.head(top_n).iterrows():
        print(f"\n[{i+1}] {row['audio']}")
        print(f"  WER: {row['wer']:.4f} | CER: {row['cer']:.4f}")
        print(f"  Reference:  {row['reference']}")
        print(f"  Hypothesis: {row['hypothesis']}")
        print("-" * 80)
    
    # Statistics
    print("\n" + "=" * 80)
    print("ERROR STATISTICS")
    print("=" * 80)
    print(f"Mean WER: {df['wer'].mean():.4f}")
    print(f"Median WER: {df['wer'].median():.4f}")
    print(f"Std WER: {df['wer'].std():.4f}")
    print(f"Min WER: {df['wer'].min():.4f}")
    print(f"Max WER: {df['wer'].max():.4f}")
    print(f"\nMean CER: {df['cer'].mean():.4f}")
    print(f"Median CER: {df['cer'].median():.4f}")
    print(f"Std CER: {df['cer'].std():.4f}")
    
    # Perfect predictions
    perfect = df[df['wer'] == 0.0]
    print(f"\nPerfect predictions (WER=0): {len(perfect)} / {len(df)} ({len(perfect)/len(df)*100:.1f}%)")
    
    # Save if requested
    if save_path:
        df_sorted.to_csv(save_path, index=False, encoding='utf-8')
        print(f"\nDetailed results saved to: {save_path}")
    
    return df_sorted

print("Error analysis function ready.")

In [51]:
# Example: Analyze errors from previous evaluation

# After running evaluate_model, analyze the errors:
error_analysis = analyze_errors(
    lm_results,
    top_n=20,
    save_path=os.path.join(WORKING_DIR, 'error_analysis.csv')
)


print("Error analysis ready to run.")

### Create Validation Report

In [52]:
def create_validation_report(baseline_results, lm_results, output_path=None):
    """
    Create a comprehensive validation report.
    
    Args:
        baseline_results: results from baseline (no LM)
        lm_results: results with language model
        output_path: path to save report (optional)
    """
    report = []
    report.append("=" * 80)
    report.append("NEWARI ASR VALIDATION REPORT")
    report.append("=" * 80)
    report.append("")
    
    # Model info
    report.append("MODEL CONFIGURATION:")
    report.append(f"  Acoustic Model: Conformer-CTC-BPE")
    report.append(f"  Language Model: {NGRAM_ORDER}-gram KenLM")
    report.append(f"  Training Corpus: {num_sentences} sentences")
    report.append(f"  Vocabulary Size: {len(vocabulary)}")
    report.append(f"  Beam Width: {BEAM_WIDTH}")
    report.append(f"  Alpha (LM weight): {ALPHA}")
    report.append(f"  Beta (word bonus): {BETA}")
    report.append("")
    
    # Results
    report.append("BASELINE RESULTS (Greedy CTC):")
    report.append(f"  WER: {baseline_results['wer']:.4f} ({baseline_results['wer']*100:.2f}%)")
    report.append(f"  CER: {baseline_results['cer']:.4f} ({baseline_results['cer']*100:.2f}%)")
    report.append(f"  Samples: {baseline_results['num_samples']}")
    report.append("")
    
    report.append("WITH LANGUAGE MODEL (Beam Search + KenLM):")
    report.append(f"  WER: {lm_results['wer']:.4f} ({lm_results['wer']*100:.2f}%)")
    report.append(f"  CER: {lm_results['cer']:.4f} ({lm_results['cer']*100:.2f}%)")
    report.append(f"  Samples: {lm_results['num_samples']}")
    report.append("")
    
    # Improvement
    wer_abs = baseline_results['wer'] - lm_results['wer']
    wer_rel = wer_abs / baseline_results['wer'] * 100
    cer_abs = baseline_results['cer'] - lm_results['cer']
    cer_rel = cer_abs / baseline_results['cer'] * 100
    
    report.append("IMPROVEMENT:")
    report.append(f"  WER: {wer_abs:+.4f} ({wer_rel:+.2f}%)")
    report.append(f"  CER: {cer_abs:+.4f} ({cer_rel:+.2f}%)")
    report.append("")
    report.append("=" * 80)
    
    report_text = "\n".join(report)
    print(report_text)
    
    if output_path:
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        print(f"\nReport saved to: {output_path}")
    
    return report_text

print("Validation report function ready.")

In [53]:
# Example: Generate final report

report = create_validation_report(
    baseline_results,
    lm_results,
    output_path=os.path.join(WORKING_DIR, 'validation_report.txt')
)


print("Report generation ready.")

## 15. Save Final Configuration

In [54]:
# Save configuration for future use
config = {
    "model_path": MODEL_PATH,
    "lm_path": BINARY_LM_PATH,
    "vocab_path": VOCAB_PATH,
    "ngram_order": NGRAM_ORDER,
    "alpha": ALPHA,
    "beta": BETA,
    "beam_width": BEAM_WIDTH,
    "corpus_size": num_sentences,
    "vocabulary_size": len(vocabulary)
}

CONFIG_PATH = os.path.join(WORKING_DIR, "lm_config.json")
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print("Configuration saved:")
print(json.dumps(config, indent=2))

## Summary

### What was created:
1. ✓ KenLM 5-gram language model trained on 40k Newari sentences
2. ✓ Binary format LM for fast loading
3. ✓ CTC decoder with language model integration
4. ✓ Transcription functions with/without LM

### Files generated:
- `newari_5gram.arpa` - ARPA format language model
- `newari_5gram.binary` - Binary format (use this for inference)
- `vocabulary.json` - Model vocabulary
- `lm_config.json` - Configuration file

### Next steps:
1. Test with your audio files
2. Fine-tune alpha/beta parameters using validation set
3. Evaluate WER improvement
4. Adjust beam width based on speed/accuracy tradeoff